In this notebook we test Vanguri et al. architecture on the I3LUNG data. 


# Set-up

In [1]:
import pandas as pd
import os
from lung_helpers import train, get_training_data, train_standard_split, hyperparameter_search

In [2]:
if not 'summary_dfs' in locals():
    print ("init summary_dfs")
    summary_dfs = {}

init summary_dfs


In [3]:
DATA_DIR = "../data"
RESULTS_DIR = "./results"

os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
df_outcomes = pd.read_csv(f"{DATA_DIR}/outcomes.csv")
df_outcomes = df_outcomes.rename(columns={'Subject': 'main_index'}).set_index('main_index')

# Find coumns with 2075 label non-null
outcome_cols_complete = []
for col in df_outcomes.columns:
    n_valid = df_outcomes[col].notna().sum()
    print(f"{col}: {n_valid} non-null")
    if n_valid == 2075:
        outcome_cols_complete.append(col)

print(f"\noutcomes with all 2075 labels: {outcome_cols_complete}")

DEATH EVENT: 2075 non-null
PROGRESSION EVENT: 2070 non-null
THERAPY END EVENT: 2075 non-null
BEST RESPONSE: 2068 non-null
IO LINE: 2075 non-null
IO IOCHT: 2075 non-null
HISTOLOGY SQUAMOUS: 2052 non-null
HISTOLOGY ADENOCARCINOMA: 2052 non-null
ORR: 2068 non-null
DCR: 2068 non-null
CBR: 2066 non-null
PFS MONTHS: 2066 non-null
OS MONTHS: 2074 non-null
TTF MONTHS: 2073 non-null
OS_6: 1972 non-null
OS_24: 1699 non-null

outcomes with all 2075 labels: ['DEATH EVENT', 'THERAPY END EVENT', 'IO LINE', 'IO IOCHT']


In [5]:
# complete cohort
df_clinical_full = pd.read_csv(f"{DATA_DIR}/rwd_processed.csv")
df_clinical_full = df_clinical_full.rename(columns={'Subject': 'main_index'})

# Cohort2 (IO LINE == 1)
df_rwd = pd.read_csv(f"{DATA_DIR}/rwd.csv")
df_rwd_cohort2 = df_rwd[df_rwd['IO LINE'] == 1].copy()

cohorts = {
    'cohort23': {
        'subjects': df_clinical_full['main_index'].values,
        'split_source': df_clinical_full
    },
    'cohort2': {
        'subjects': df_rwd_cohort2['Subject'].values,
        'split_source': df_rwd_cohort2.rename(columns={'Subject': 'main_index'})
    }
}

all_splits = {}

for cohort_name, cohort_info in cohorts.items():
    print(f"\n{'='*50}")
    print(f"Processing {cohort_name}")
    print(f"{'='*50}")
    
    # filter df_clinical_full for the subjects in the cohort
    cohort_subjects = cohort_info['subjects']
    df_clinical = df_clinical_full[df_clinical_full['main_index'].isin(cohort_subjects)].copy()
    
    # extract splits
    split_df = cohort_info['split_source']
    train_df = split_df[split_df['SET'] == 'TRAIN']
    test_df = split_df[split_df['SET'] == 'TEST']
    ext_val_df = split_df[split_df['SET'] == 'EXVAL']
    
    train_px = train_df['main_index'].values
    test_px = test_df['main_index'].values
    ext_val_px = ext_val_df['main_index'].values
    
    print(f"=== INITIAL SPLIT {cohort_name} ===")
    print(f"Train: {len(train_px)}, Test: {len(test_px)}, ExtVal: {len(ext_val_px)}")
    
    # extract predefined folds (exclude UOC from training centers)
    predefined_folds = train_df[train_df['CENTER'] != 'UOC'][['main_index', 'CENTER']].rename(columns={'CENTER': 'fold'})
    predefined_folds = predefined_folds.set_index('main_index')
    
    print(f"Train Centers (per CV): {predefined_folds['fold'].unique()}")
    print(f"Patients for center:\n{predefined_folds['fold'].value_counts()}")
    
    # prepare clinical dataframe
    df_clinical = df_clinical.set_index('main_index')
    df_clinical = df_clinical.drop(columns=['CENTER', 'SET'], errors='ignore')
    
    # save for later use
    all_splits[cohort_name] = {
        'df_clinical': df_clinical,
        'train_px': train_px,
        'test_px': test_px,
        'ext_val_px': ext_val_px,
        'predefined_folds': predefined_folds
    }


Processing cohort23
=== INITIAL SPLIT cohort23 ===
Train: 1550, Test: 274, ExtVal: 251
Train Centers (per CV): ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
Patients for center:
fold
INT     582
GHD     365
SZMC    232
MH      196
VHIO    175
Name: count, dtype: int64

Processing cohort2
=== INITIAL SPLIT cohort2 ===
Train: 1046, Test: 181, ExtVal: 210
Train Centers (per CV): ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
Patients for center:
fold
INT     386
GHD     231
SZMC    193
MH      164
VHIO     72
Name: count, dtype: int64


In [6]:
df_pyrad = pd.read_csv(f"{DATA_DIR}/pyradiomics.csv")
df_pyrad = df_pyrad.rename(columns={'Subject': 'main_index'})

print(f"total patients: {len(df_pyrad)}")

# Set index e drop colonne
df_pyrad = df_pyrad.set_index('main_index')
df_pyrad = df_pyrad.drop(columns=['CENTER', 'SET'], errors='ignore')

print(f"pyradiomics features: {len(df_pyrad.columns)}")

total patients: 877
pyradiomics features: 128


In [7]:
df_fmrad = pd.read_csv(f"{DATA_DIR}/fmrad.csv")
df_fmrad = df_fmrad.rename(columns={'Subject': 'main_index'})

print(f"total patients: {len(df_fmrad)}")
# Set index e drop columns
df_fmrad = df_fmrad.set_index('main_index')
df_fmrad = df_fmrad.drop(columns=['CENTER', 'SET'], errors='ignore')

print(f"fmradiomics features: {len(df_fmrad.columns)}")

total patients: 896
fmradiomics features: 4096


In [8]:
df_pathology = pd.read_csv(f"{DATA_DIR}/digital_pathology_processed.csv")  
df_pathology = df_pathology.rename(columns={'Subject': 'main_index'})  

print(f"total patients: {len(df_pathology)}")

# Set index e drop columns
df_pathology = df_pathology.set_index('main_index')
df_pathology = df_pathology.drop(columns=['CENTER', 'SET'], errors='ignore')  

print(f"pathology features: {len(df_pathology.columns)}")

total patients: 846
pathology features: 768


In [9]:
df_genomics = pd.read_csv(f"{DATA_DIR}/genomics_processed.csv") 
df_genomics = df_genomics.rename(columns={'Subject': 'main_index'})  

print(f"total patients: {len(df_genomics)}")

# Set index e drop columns
df_genomics = df_genomics.set_index('main_index')
df_genomics = df_genomics.drop(columns=['CENTER', 'SET'], errors='ignore')  #

print(f"genomics features: {len(df_genomics.columns)}")

total patients: 1705
genomics features: 4


In [10]:
df_outcomes = pd.read_csv(f"{DATA_DIR}/outcomes.csv")
df_outcomes = df_outcomes.rename(columns={'Subject': 'main_index'})
df_outcomes = df_outcomes.set_index('main_index')
df_outcomes = df_outcomes[~df_outcomes.index.duplicated()]
print("total patients:", df_outcomes.index.nunique())
print(f"total outcomes: {len(df_outcomes.columns)}")

total patients: 2075
total outcomes: 16


# Hyperparam search

default hyperparam from original paper

In [11]:
# for multimodal models, attention gate
model_params = {'epochs':125, 'lr':0.01, 'alpha':0.001, 'beta':0.0, 'cross_modality_enabled':False}
# for unimodal models, no attention gate
model_params_gate_off = {'epochs':125, 'lr':0.01, 'alpha':0.001, 'beta':0.0, 'cross_modality_enabled':False, 'attention_gate_enabled':False}

hyperparam grid

In [12]:
hyperparam_grid = {
    'epochs': [100, 125, 150],
    'lr': [0.001, 0.01, 0.1],
    'alpha': [0.0001, 0.001, 0.01],  # L2 weight
    'beta': [0.0, 0.001, 0.01]       # AR2 weight
}

In [13]:
# Prepare data for cohort23, OS_6
cohort_name = 'cohort23'
outcome_col = 'OS_6'

df_clinical_clean = all_splits[cohort_name]['df_clinical'].copy()
train_px = all_splits[cohort_name]['train_px']

# Filtra solo train set
df_clinical_train = df_clinical_clean.loc[train_px]

cohort_subjects = df_clinical_train.index
df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()

df_dyam = df_clinical_train.copy()
df_dyam['label'] = df_outcomes[outcome_col]
df_dyam = df_dyam.dropna(subset=['label'])

X_clin = df_dyam.drop(columns=['label']).fillna(0)
X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)

modality_mask = pd.DataFrame({
    'clinical': 1,
    'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
    'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
    'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
}, index=df_dyam.index)

df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)

# L1 filter
df_rad_for_filter = X_rad.reset_index()
df_rad_for_filter['job_tag'] = 'filtered-radiomics'
df_rad_for_filter['site'] = 'PC'
df_rad_for_filter['lesion_index'] = 1

dfs_filters = {
    1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
}

# Run hyperparameter search
best_params, search_results = hyperparameter_search(
    [X_clin, X_rad, X_path, X_gen],
    modality_mask, 
    df_out,
    dfs_filters,
    hyperparam_grid
)

# Save results
search_results.to_csv('results/hyperparam_search_results_cohort23_OS_6.csv', index=False)

print(f"\nUsing best parameters for all models: {best_params}")

Testing 81 combinations
Train: 1325, Validation: 148

Train samples: 1325
Test samples: 148
Applying L1 filter on modality position 1
Selected 17 outlier-stable features


Training 100 epochs with 6 batches
Final output - min: -0.9600, max: 0.8659, mean: 0.0176
Training mode - mu: -0.0101, std: 0.3353
Eval mode - using mu: -0.0101, std: 0.3353
Eval mode - using mu: -0.0101, std: 0.3353

Test AUC: 0.648 (95% CI: 0.548-0.747)
epochs=100, lr=0.001, alpha=0.0001, beta=0.0 -> Val AUC: 0.658 (95% CI: 0.629-0.688)
  -> New best (no CI overlap)
Train samples: 1325
Test samples: 148
Applying L1 filter on modality position 1
Selected 17 outlier-stable features
Training 100 epochs with 6 batches
Final output - min: -0.9657, max: 0.9023, mean: 0.0462
Training mode - mu: 0.0366, std: 0.3271
Eval mode - using mu: 0.0366, std: 0.3271
Eval mode - using mu: 0.0366, std: 0.3271

Test AUC: 0.649 (95% CI: 0.554-0.744)
epochs=100, lr=0.001, alpha=0.0001, beta=0.001 -> Val AUC: 0.672 (95% CI: 0.644-0.701)
  -> New best (no CI overlap)
Train samples: 1325
Test samples: 148
Applying L1 filter on modality position 1
Selected 17 outlier-stable features
Training 100 epochs with 6 

In [14]:
# for multimodal models, attention gate
model_params = {'epochs': 125, 'lr': 0.1, 'alpha': 0.01, 'beta': 0.0, 'cross_modality_enabled': False}
# for unimodal models, no attention gate
model_params_gate_off = {'epochs': 125, 'lr': 0.1, 'alpha': 0.01, 'beta': 0.0, 'cross_modality_enabled': False, 'attention_gate_enabled': False}

# Unimodal clinical

cross validation unimodal

In [15]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training models for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        X_dyam = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_dyam.isna().sum().sum()}")
        
        modality_mask_single = pd.DataFrame({
            'clinical': X_dyam.notna().any(axis=1).astype(int)
        }, index=X_dyam.index)
        print(f"Modality mask created with {modality_mask_single['clinical'].sum()} samples having clinical data")
        
        X_dyam_filled = X_dyam.fillna(0)
        print(f"After filling missing values: {X_dyam_filled.isna().sum().sum()} missing values")
        
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        common_index = X_dyam_filled.index & modality_mask_single.index & df_out.index
        X_dyam_filled = X_dyam_filled.loc[common_index]
        modality_mask_single = modality_mask_single.loc[common_index]
        df_out = df_out.loc[common_index]
        
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        cv_index = predefined_folds_aligned.index
        X_dyam_cv = X_dyam_filled.loc[cv_index]
        modality_mask_cv = modality_mask_single.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'], _ = train(
            modality_list_in=[X_dyam_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter={},
            model_params=model_params_gate_off,
            predefined_folds=predefined_folds_aligned
        )
        
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'].to_csv(
            f'results/cv_rwd_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training models for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Modality mask created with 1699 samples having clinical data
After filling missing values: 0 missing values
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC']


Processing fold 1: 921 train samples, 362 validation samples
Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.1519
Training mode - mu: -0.0023, std: 0.6669
Eval mode - using mu: -0.0023, std: 0.6669
 20%|██        | 1/5 [00:01<00:05,  1.37s/it]
Processing fold 2: 773 train samples, 510 validation samples
Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.3603
Training mode - mu: -0.5279, std: 0.7488
Eval mode - using mu: -0.5279, std: 0.7488
 40%|████      | 2/5 [00:02<00:03,  1.19s/it]
Processing fold 3: 1175 train samples, 108 validation samples
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.3143
Training mode - mu: -0.2717, std: 0.6486
Eval mode - using mu: -0.2717, std: 0.6486
 60%|██████    | 3/5 [00:04<00:02,  1.43s/it]
Processing fold 4: 1144 train samples, 139 validation samples
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2496

train/test/ext-val unimodal

In [16]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training models for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Feature matrix
        X_dyam = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_dyam.isna().sum().sum()}")
        
        # Modality mask
        modality_mask_single = pd.DataFrame({
            'clinical': X_dyam.notna().any(axis=1).astype(int)
        }, index=X_dyam.index)
        print(f"Modality mask created with {modality_mask_single['clinical'].sum()} samples having clinical data")
        
        # Fill missing
        X_dyam_filled = X_dyam.fillna(0)
        print(f"After filling missing values: {X_dyam_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_dyam_filled.index & modality_mask_single.index & df_out.index
        X_dyam_filled = X_dyam_filled.loc[common_index]
        modality_mask_single = modality_mask_single.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # Train
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_dyam_filled],
            modality_mask=modality_mask_single,
            outcomes=df_out,
            l1_dfs_filter={},
            model_params=model_params_gate_off,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Salva predizioni
        summary_dfs[f'{cohort_name}_Clinical_{outcome_col}'].to_csv(
            f'results/standard_rwd_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training models for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Modality mask created with 1699 samples having clinical data
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.0819
Training mode - mu: 0.2660, std: 0.8447
Eval mode - using mu: 0.2660, std: 0.8447
Eval mode - using mu: 0.2660, std: 0.8447

Test AUC: 0.703 (95% CI: 0.626-0.781)
Eval mode - using mu: 0.2660, std: 0.8447
External Validation AUC: 0.533 (95% CI: 0.481-0.584)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Modality mask created with 1972

# Bimodal Clinical + Radiomics

bimodal clinical + pyrad cross validation

In [17]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']

    print(f"Clinical samples for {cohort_name}: {len(df_clinical_clean)}")
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()

    print(f"Radiology samples for {cohort_name}: {len(df_radiology_clean)}")
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad.fillna(0)
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align / saftey check 
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        print(f"Common index samples: {len(common_index)}")
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]

        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")

        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue


        # Filter for CV 
        X_clin_cv = X_clin_filled.loc[predefined_folds_aligned.index]
        X_rad_cv = X_rad_filled.loc[predefined_folds_aligned.index]
        modality_mask_cv = modality_mask_bimodal.loc[predefined_folds_aligned.index]
        df_out_cv = df_out.loc[predefined_folds_aligned.index]

        
        # L1 filter
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_rad_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'].to_csv(
            f'results/cv_rwd_pyrad_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models for cohort23
Clinical samples for cohort23: 2075
Radiology samples for cohort23: 877

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
Common index samples: 1699
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<?, ?it/s


Processing fold 1: 921 train samples, 362 validation samples
 Applying L1 filter on modality position 1
Selected 17 outlier-stable features
Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2079
Training mode - mu: -0.1244, std: 0.6344
Eval mode - using mu: -0.1244, std: 0.6344
 20%|██        | 1/5 [00:01<00:07,  1.94s/it]
Processing fold 2: 773 train samples, 510 validation samples
 Applying L1 filter on modality position 1
Selected 15 outlier-stable features
Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2381
Training mode - mu: -0.2002, std: 0.7781
Eval mode - using mu: -0.2002, std: 0.7781
 40%|████      | 2/5 [00:03<00:05,  1.83s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 13 outlier-stable features
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2271
Training mode - mu: -0.0713, std: 0.6825
Eval

bimodal clinical + pyrad train/test/ext_val split 

In [18]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filter setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled],
            modality_mask=modality_mask_bimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_rad_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad_{outcome_col}'].to_csv(
            f'results/standard_rwd_pyrad_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 9 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2847
Training mode - mu: -0.2971, std: 0.7669
Eval mode - using mu: -0.2971, std: 0.7669
Eval mode - using mu: -0.2971, std: 0.7669

Test AUC: 0.628 (95% CI: 0.544-0.711)
Eval mode - using mu: -0.2971, std: 0.7669
External Validation AUC: 0.527 (95% CI: 0.448-0.607)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, i

bimodal clinical + fmrad cross validation

In [19]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (FM radiology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        modality_mask_cv = modality_mask_bimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filter
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_rad_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'].to_csv(
            f'results/cv_rwd_radfm_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (FM radiology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 798
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<?, ?it/s]
Processing fold 1: 921 train samples, 362

bimodal clinical + fmrad train/test/ext_val

In [20]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (FM radiology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology (allineato con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_bimodal['radiology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filter setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        dfs_rad_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled],
            modality_mask=modality_mask_bimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_rad_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM_{outcome_col}'].to_csv(
            f'results/standard_rwd_radfm_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (FM radiology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 798
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 3694 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2394
Training mode - mu: -0.1067, std: 0.7556
Eval mode - using mu: -0.1067, std: 0.7556
Eval mode - using mu: -0.1067, std: 0.7556

Test AUC: 0.587 (95% CI: 0.499-0.674)
Eval mode - using mu: -0.1067, std: 0.7556
External Validation AUC: 0.464 (95% CI: 0.382-0.546)


--- Processing outcome: OS_6 for cohort23 ---
Processing

# Bimodal Clinical + Digital Pathology

bimodal clinical + dp cross validation

In [21]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (pathology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features pathology (allineato con reindex)
        X_dp = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with pathology data: {modality_mask_bimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_dp_filled = X_dp
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_dp_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_dp_filled = X_dp_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_dp_cv = X_dp_filled.loc[cv_index]
        modality_mask_cv = modality_mask_bimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_dp_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter={},
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'].to_csv(
            f'results/cv_rwd_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (pathology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with pathology data: 708
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<?, ?it/s]
Processing fold 1: 921 train samples, 362 va

Final output - min: -1.0000, max: 1.0000, mean: -0.2031
Training mode - mu: -0.0768, std: 0.6854
Eval mode - using mu: -0.0768, std: 0.6854
100%|██████████| 5/5 [00:25<00:00,  5.09s/it]

Overall AUC calculated on 1283 samples (pos=324, neg=959)
AUC = 0.687 (95% CI: 0.652-0.723)

--- Processing outcome: OS_6 for cohort23 ---
Patients with OS_6 label: 1972
Total patients: 1972
Patients with clinical data: 1972
Patients with pathology data: 820
Predefined folds after alignment: 1473 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1473 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 365 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (1108 samples)
Fold INT (test): 567 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (906 samples)
Fold MH (test): 165 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1308 samples)
Fold SZMC (test): 202 samples | Traini

bimodal clinical + dp standard

In [22]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training bimodal models (pathology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features pathology (allineato con reindex)
        X_dp = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_bimodal = pd.DataFrame({
            'clinical': 1,
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_bimodal['clinical'].sum()}")
        print(f"Patients with pathology data: {modality_mask_bimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_dp_filled = X_dp
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_dp_filled.index & modality_mask_bimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_dp_filled = X_dp_filled.loc[common_index]
        modality_mask_bimodal = modality_mask_bimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # Train bimodal
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_dp_filled],
            modality_mask=modality_mask_bimodal,
            outcomes=df_out,
            l1_dfs_filter={},
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+DP_{outcome_col}'].to_csv(
            f'results/standard_rwd_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training bimodal models (pathology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with pathology data: 708
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Training 125 epochs with 6 batches


Final output - min: -1.0000, max: 1.0000, mean: -0.1376
Training mode - mu: 0.0891, std: 0.7603
Eval mode - using mu: 0.0891, std: 0.7603
Eval mode - using mu: 0.0891, std: 0.7603

Test AUC: 0.655 (95% CI: 0.570-0.739)
Eval mode - using mu: 0.0891, std: 0.7603
External Validation AUC: 0.574 (95% CI: 0.502-0.647)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with pathology data: 820
After filling missing values: 0 missing values
After filtering - Train: 1473, Test: 259, ExtVal: 240
Train samples: 1473
Test samples: 259
External validation samples: 240
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: 0.0452
Training mode - mu: -0.0112, std: 0.5933
Eval mode - using mu: -0.0112, std: 0.5933
Eval mode - using mu: -0.0112, std: 0.5933

Test AUC: 0.664 (95% CI: 0.59

# Trimodal Clinical + Radiomics + Digital Pathology

trimodal clinical + pyrad + dp cross validation

In [23]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+radiology+pathology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        modality_mask_cv = modality_mask_trimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'].to_csv(
            f'results/cv_rwd_pyrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+radiology+pathology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
Patients with pathology data: 708
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:00<

Final output - min: -1.0000, max: 1.0000, mean: -0.1633
Training mode - mu: 0.0411, std: 0.7366
Eval mode - using mu: 0.0411, std: 0.7366
 40%|████      | 2/5 [00:09<00:13,  4.62s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 13 outlier-stable features
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2119
Training mode - mu: -0.1081, std: 0.6438
Eval mode - using mu: -0.1081, std: 0.6438
 60%|██████    | 3/5 [00:15<00:10,  5.33s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 Applying L1 filter on modality position 1
Selected 14 outlier-stable features
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2495
Training mode - mu: -0.0887, std: 0.6490
Eval mode - using mu: -0.0887, std: 0.6490
 80%|████████  | 4/5 [00:22<00:05,  5.84s/it]
Processing fold 5: 1119 train samples, 164 validation samples
 Applying L1 filter on modalit

trimodal clinical + pyrad + dp train/test/extval split

In [24]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+radiology+pathology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled],
            modality_mask=modality_mask_trimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path_{outcome_col}'].to_csv(
            f'results/standard_rwd_pyrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+radiology+pathology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
Patients with pathology data: 708
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 9 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.1985
Training mode - mu: -0.0342, std: 0.6821
Eval mode - using mu: -0.0342, std: 0.6821
Eval mode - using mu: -0.0342, std: 0.6821

Test AUC: 0.583 (95% CI: 0.495-0.670)
Eval mode - using mu: -0.0342, std: 0.6821
External Validation AUC: 0.508 (95% CI: 0.425-0.592)


--- Proc

Final output - min: -1.0000, max: 1.0000, mean: 0.0474
Training mode - mu: 0.0842, std: 0.5743
Eval mode - using mu: 0.0842, std: 0.5743
Eval mode - using mu: 0.0842, std: 0.5743

Test AUC: 0.695 (95% CI: 0.631-0.760)
Eval mode - using mu: 0.0842, std: 0.5743
External Validation AUC: 0.536 (95% CI: 0.464-0.608)


--- Processing outcome: ORR for cohort23 ---
Processing outcome: ORR, initial samples: 2075
After dropping missing labels: 2068 samples
Initial missing values in features: 0
Total patients: 2068
Patients with clinical data: 2068
Patients with radiology data: 877
Patients with pathology data: 844
After filling missing values: 0 missing values
After filtering - Train: 1544, Test: 273, ExtVal: 251
Train samples: 1544
Test samples: 273
External validation samples: 251
Applying L1 filter on modality position 1
Selected 9 outlier-stable features
Training 125 epochs with 7 batches
Final output - min: -0.9999, max: 1.0000, mean: 0.0552
Training mode - mu: 0.1198, std: 0.5242
Eval mode

trimodal clinical + fmrad + dp cross val

In [25]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+FM radiology+pathology) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        modality_mask_cv = modality_mask_trimodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'].to_csv(
            f'results/cv_rwd_fmrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+FM radiology+pathology) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 798
Patients with pathology data: 708
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC'] (1119 samples)
  0%|          | 0/5 [00:

Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2018
Training mode - mu: -0.1094, std: 0.6152
Eval mode - using mu: -0.1094, std: 0.6152
 20%|██        | 1/5 [00:44<02:58, 44.60s/it]
Processing fold 2: 773 train samples, 510 validation samples
 Applying L1 filter on modality position 1
Selected 3781 outlier-stable features
Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2759
Training mode - mu: -0.2051, std: 0.6842
Eval mode - using mu: -0.2051, std: 0.6842
 40%|████      | 2/5 [01:29<02:14, 44.78s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 3709 outlier-stable features
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2151
Training mode - mu: -0.0274, std: 0.6333
Eval mode - using mu: -0.0274, std: 0.6333
 60%|██████    | 3/5 [02:21<01:36, 48.10s/it]
Processing fold 4: 1144 train samples, 139 validatio

trimodal clinical + fmrad + dp train/test/extval split

In [26]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training trimodal models (clinical+FM radiology+pathology, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology e pathology per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology e pathology (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_trimodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_trimodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_trimodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_trimodal['pathology'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & modality_mask_trimodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        modality_mask_trimodal = modality_mask_trimodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train trimodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled],
            modality_mask=modality_mask_trimodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path_{outcome_col}'].to_csv(
            f'results/standard_rwd_fmrad_dp_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training trimodal models (clinical+FM radiology+pathology, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 798
Patients with pathology data: 708
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 3694 outlier-stable features


Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.1397
Training mode - mu: -0.1653, std: 0.6657
Eval mode - using mu: -0.1653, std: 0.6657
Eval mode - using mu: -0.1653, std: 0.6657

Test AUC: 0.706 (95% CI: 0.623-0.789)
Eval mode - using mu: -0.1653, std: 0.6657
External Validation AUC: 0.469 (95% CI: 0.395-0.543)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 877
Patients with pathology data: 820
After filling missing values: 0 missing values
After filtering - Train: 1473, Test: 259, ExtVal: 240
Train samples: 1473
Test samples: 259
External validation samples: 240
Applying L1 filter on modality position 1
Selected 3636 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: 0.0919
Trai

# Multimodal Clinical + Radiomics + Digital Pathology + Genomics

multimodal clinical + pyrad + dp + genomics cross val

In [27]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+radiology+pathology+genomics) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        X_gen_cv = X_gen_filled.loc[cv_index]
        modality_mask_cv = modality_mask_quadmodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv, X_gen_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'].to_csv(
            f'results/cv_rwd_pyrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+radiology+pathology+genomics) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
Patients with pathology data: 708
Patients with genomics data: 1363
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZMC']

Final output - min: -0.9975, max: 0.9958, mean: 0.1958
Training mode - mu: 0.1568, std: 0.5729
Eval mode - using mu: 0.1568, std: 0.5729
 20%|██        | 1/5 [00:08<00:35,  8.95s/it]
Processing fold 2: 906 train samples, 567 validation samples
 Applying L1 filter on modality position 1
Selected 17 outlier-stable features
Training 125 epochs with 4 batches
Final output - min: -0.9985, max: 0.9988, mean: 0.0510
Training mode - mu: -0.0161, std: 0.5522
Eval mode - using mu: -0.0161, std: 0.5522
 40%|████      | 2/5 [00:15<00:23,  7.75s/it]
Processing fold 3: 1308 train samples, 165 validation samples
 Applying L1 filter on modality position 1
Selected 13 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -0.9996, max: 0.9986, mean: 0.0070
Training mode - mu: -0.0255, std: 0.5703
Eval mode - using mu: -0.0255, std: 0.5703
 60%|██████    | 3/5 [00:23<00:15,  7.52s/it]
Processing fold 4: 1271 train samples, 202 validation samples
 Applying L1 filter on modality po

multimodal clinical + pyrad + dp + genomics train/test/ext_val

In [28]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+radiology+pathology+genomics, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_pyrad[df_pyrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}},
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled, X_gen_filled],
            modality_mask=modality_mask_quadmodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+Rad+Path+Gen_{outcome_col}'].to_csv(
            f'results/standard_rwd_pyrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+radiology+pathology+genomics, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 782
Patients with pathology data: 708
Patients with genomics data: 1363
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 9 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.1918
Training mode - mu: -0.0363, std: 0.7103
Eval mode - using mu: -0.0363, std: 0.7103
Eval mode - using mu: -0.0363, std: 0.7103

Test AUC: 0.603 (95% CI: 0.521-0.685)
Eval mode - using mu: -0.0363, std: 0.7103
External Validation

Final output - min: -0.9979, max: 0.9983, mean: -0.0411
Training mode - mu: -0.1587, std: 0.5782
Eval mode - using mu: -0.1587, std: 0.5782
Eval mode - using mu: -0.1587, std: 0.5782

Test AUC: 0.663 (95% CI: 0.554-0.771)
Eval mode - using mu: -0.1587, std: 0.5782
External Validation AUC: 0.567 (95% CI: 0.475-0.659)


--- Processing outcome: DCR for cohort2 ---
Processing outcome: DCR, initial samples: 1437
After dropping missing labels: 1431 samples
Initial missing values in features: 0
Total patients: 1431
Patients with clinical data: 1431
Patients with radiology data: 562
Patients with pathology data: 634
Patients with genomics data: 1221
After filling missing values: 0 missing values
After filtering - Train: 1041, Test: 180, ExtVal: 210
Train samples: 1041
Test samples: 180
External validation samples: 210
Applying L1 filter on modality position 1
Selected 10 outlier-stable features
Training 125 epochs with 5 batches
Final output - min: -0.9988, max: 0.9982, mean: -0.0692
Training 

multimodal clinical + radfm + dp + genomics cross val

In [29]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+FM radiology+pathology+genomics) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    predefined_folds = cohort_data['predefined_folds']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        df_dyam = df_dyam.dropna(subset=['label'])
        
        print(f"Patients with {outcome_col} label: {len(df_dyam)}")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Align predefined_folds
        predefined_folds_aligned = predefined_folds.loc[predefined_folds.index.isin(common_index)]
        
        print(f"Predefined folds after alignment: {len(predefined_folds_aligned)} samples")
        print(f"Folds available: {predefined_folds_aligned['fold'].unique() if len(predefined_folds_aligned) > 0 else 'NONE'}")
        
        if len(predefined_folds_aligned) == 0:
            print(f"WARNING: No predefined folds available for {cohort_name} {outcome_col}, skipping CV")
            continue
        
        # Filter for CV (exclude UOC)
        cv_index = predefined_folds_aligned.index
        X_clin_cv = X_clin_filled.loc[cv_index]
        X_rad_cv = X_rad_filled.loc[cv_index]
        X_path_cv = X_path_filled.loc[cv_index]
        X_gen_cv = X_gen_filled.loc[cv_index]
        modality_mask_cv = modality_mask_quadmodal.loc[cv_index]
        df_out_cv = df_out.loc[cv_index]
        
        print(f"After excluding UOC: {len(cv_index)} samples for CV")
        
        # L1 filters
        df_rad_for_filter = X_rad_cv.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters_aligned = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'], _ = train(
            modality_list_in=[X_clin_cv, X_rad_cv, X_path_cv, X_gen_cv],
            modality_mask=modality_mask_cv,
            outcomes=df_out_cv,
            l1_dfs_filter=dfs_filters_aligned,
            model_params=model_params,
            predefined_folds=predefined_folds_aligned
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'].to_csv(
            f'results/cv_rwd_fmrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+FM radiology+pathology+genomics) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Patients with OS_24 label: 1699
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 798
Patients with pathology data: 708
Patients with genomics data: 1363
Predefined folds after alignment: 1283 samples
Folds available: ['GHD' 'INT' 'MH' 'SZMC' 'VHIO']
After excluding UOC: 1283 samples for CV

Starting cross-validation training
predefined_folds: True
Using predefined folds for cross-validation
Fold GHD (test): 362 samples | Training on folds: ['INT' 'MH' 'SZMC' 'VHIO'] (921 samples)
Fold INT (test): 510 samples | Training on folds: ['GHD' 'MH' 'SZMC' 'VHIO'] (773 samples)
Fold MH (test): 108 samples | Training on folds: ['GHD' 'INT' 'SZMC' 'VHIO'] (1175 samples)
Fold SZMC (test): 139 samples | Training on folds: ['GHD' 'INT' 'MH' 'VHIO'] (1144 samples)
Fold VHIO (test): 164 samples | Training on folds: ['GHD' 'INT' 'MH' 'SZM

Final output - min: -1.0000, max: 0.9999, mean: -0.1762
Training mode - mu: -0.0714, std: 0.6181
Eval mode - using mu: -0.0714, std: 0.6181
 20%|██        | 1/5 [00:46<03:06, 46.74s/it]
Processing fold 2: 773 train samples, 510 validation samples
 Applying L1 filter on modality position 1
Selected 3781 outlier-stable features
Training 125 epochs with 4 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.3444
Training mode - mu: -0.3712, std: 0.6219
Eval mode - using mu: -0.3712, std: 0.6219
 40%|████      | 2/5 [01:33<02:20, 46.87s/it]
Processing fold 3: 1175 train samples, 108 validation samples
 Applying L1 filter on modality position 1
Selected 3709 outlier-stable features
Training 125 epochs with 5 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.2814
Training mode - mu: -0.1697, std: 0.5977
Eval mode - using mu: -0.1697, std: 0.5977
 60%|██████    | 3/5 [02:31<01:43, 51.77s/it]
Processing fold 4: 1144 train samples, 139 validation samples
 Applying L1 filter on mo

multimodal clinical + radfm + dp + genomics train/test/ext_val

In [30]:
summary_dfs = {}

for cohort_name, cohort_data in all_splits.items():
    print(f"\n{'='*50}")
    print(f"Training quadmodal models (clinical+FM radiology+pathology+genomics, standard split) for {cohort_name}")
    print(f"{'='*50}")
    
    df_clinical_clean = cohort_data['df_clinical'].copy()
    train_px = cohort_data['train_px']
    test_px = cohort_data['test_px']
    ext_val_px = cohort_data['ext_val_px']
    
    # Filtra radiology, pathology e genomics per cohort
    cohort_subjects = df_clinical_clean.index
    df_radiology_clean = df_fmrad[df_fmrad.index.isin(cohort_subjects)].copy()
    df_pathology_clean = df_pathology[df_pathology.index.isin(cohort_subjects)].copy()
    df_genomics_clean = df_genomics[df_genomics.index.isin(cohort_subjects)].copy()
    
    for outcome_col in ['OS_24', 'OS_6', 'DCR', 'ORR', 'CBR']:
        print(f"\n--- Processing outcome: {outcome_col} for {cohort_name} ---")
        
        # Prepare data with label
        df_dyam = df_clinical_clean.copy()
        df_dyam['label'] = df_outcomes[outcome_col]
        print(f"Processing outcome: {outcome_col}, initial samples: {len(df_dyam)}")
        df_dyam = df_dyam.dropna(subset=['label'])
        print(f"After dropping missing labels: {len(df_dyam)} samples")
        
        # Features clinical
        X_clin = df_dyam.drop(columns=['label'])
        print(f"Initial missing values in features: {X_clin.isna().sum().sum()}")
        
        # Features radiology, pathology, genomics (allineati con reindex)
        X_rad = df_radiology_clean.reindex(df_dyam.index).fillna(0)
        X_path = df_pathology_clean.reindex(df_dyam.index).fillna(0)
        X_gen = df_genomics_clean.reindex(df_dyam.index).fillna(0)
        
        # Create modality mask
        modality_mask_quadmodal = pd.DataFrame({
            'clinical': 1,
            'radiology': df_dyam.index.isin(df_radiology_clean.index).astype(int),
            'pathology': df_dyam.index.isin(df_pathology_clean.index).astype(int),
            'genomics': df_dyam.index.isin(df_genomics_clean.index).astype(int)
        }, index=df_dyam.index)
        
        print(f"Total patients: {len(df_dyam)}")
        print(f"Patients with clinical data: {modality_mask_quadmodal['clinical'].sum()}")
        print(f"Patients with radiology data: {modality_mask_quadmodal['radiology'].sum()}")
        print(f"Patients with pathology data: {modality_mask_quadmodal['pathology'].sum()}")
        print(f"Patients with genomics data: {modality_mask_quadmodal['genomics'].sum()}")
        
        # Fill missing
        X_clin_filled = X_clin.fillna(0)
        X_rad_filled = X_rad
        X_path_filled = X_path
        X_gen_filled = X_gen
        print(f"After filling missing values: {X_clin_filled.isna().sum().sum()} missing values")
        
        # Outcomes
        df_out = pd.DataFrame({'label': df_dyam['label']}, index=df_dyam.index)
        
        # Align
        common_index = X_clin_filled.index & X_rad_filled.index & X_path_filled.index & X_gen_filled.index & modality_mask_quadmodal.index & df_out.index
        X_clin_filled = X_clin_filled.loc[common_index]
        X_rad_filled = X_rad_filled.loc[common_index]
        X_path_filled = X_path_filled.loc[common_index]
        X_gen_filled = X_gen_filled.loc[common_index]
        modality_mask_quadmodal = modality_mask_quadmodal.loc[common_index]
        df_out = df_out.loc[common_index]
        
        # Intersecare con gli split
        train_px_filtered = [px for px in train_px if px in common_index]
        test_px_filtered = [px for px in test_px if px in common_index]
        ext_val_px_filtered = [px for px in ext_val_px if px in common_index]
        
        print(f"After filtering - Train: {len(train_px_filtered)}, Test: {len(test_px_filtered)}, ExtVal: {len(ext_val_px_filtered)}")
        
        # Verifica che non siano vuoti
        if len(train_px_filtered) == 0:
            print(f"WARNING: No training samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(test_px_filtered) == 0:
            print(f"WARNING: No test samples for {cohort_name} {outcome_col}, skipping")
            continue
        if len(ext_val_px_filtered) == 0:
            print("WARNING: No external validation samples, setting ext_val_set=None")
            ext_val_px_filtered = None
        
        # L1 filters setup
        df_rad_for_filter = X_rad_filled.reset_index()
        df_rad_for_filter['job_tag'] = 'filtered-radiomics'
        df_rad_for_filter['site'] = 'PC'
        df_rad_for_filter['lesion_index'] = 1
        
        dfs_filters = {
            1: {'l1_selection_df': df_rad_for_filter, 'kwargs': {'l1_strength': 0.1}}
        }
        
        # Train quadmodal
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'], _ = train_standard_split(
            modality_list_in=[X_clin_filled, X_rad_filled, X_path_filled, X_gen_filled],
            modality_mask=modality_mask_quadmodal,
            outcomes=df_out,
            l1_dfs_filter=dfs_filters,
            model_params=model_params,
            train_set=train_px_filtered,
            test_set=test_px_filtered,
            ext_val_set=ext_val_px_filtered
        )
        
        # Save
        summary_dfs[f'{cohort_name}_Clinical+RadFM+Path+Gen_{outcome_col}'].to_csv(
            f'results/standard_rwd_fmrad_path_gen_{cohort_name}_{outcome_col}.csv',
            index=True
        )


Training quadmodal models (clinical+FM radiology+pathology+genomics, standard split) for cohort23

--- Processing outcome: OS_24 for cohort23 ---
Processing outcome: OS_24, initial samples: 2075
After dropping missing labels: 1699 samples
Initial missing values in features: 0
Total patients: 1699
Patients with clinical data: 1699
Patients with radiology data: 798
Patients with pathology data: 708
Patients with genomics data: 1363
After filling missing values: 0 missing values
After filtering - Train: 1283, Test: 226, ExtVal: 190
Train samples: 1283
Test samples: 226
External validation samples: 190
Applying L1 filter on modality position 1
Selected 3694 outlier-stable features


Training 125 epochs with 6 batches
Final output - min: -1.0000, max: 1.0000, mean: -0.3423
Training mode - mu: -0.3938, std: 0.6197
Eval mode - using mu: -0.3938, std: 0.6197
Eval mode - using mu: -0.3938, std: 0.6197

Test AUC: 0.631 (95% CI: 0.549-0.714)
Eval mode - using mu: -0.3938, std: 0.6197
External Validation AUC: 0.485 (95% CI: 0.399-0.570)


--- Processing outcome: OS_6 for cohort23 ---
Processing outcome: OS_6, initial samples: 2075
After dropping missing labels: 1972 samples
Initial missing values in features: 0
Total patients: 1972
Patients with clinical data: 1972
Patients with radiology data: 877
Patients with pathology data: 820
Patients with genomics data: 1615
After filling missing values: 0 missing values
After filtering - Train: 1473, Test: 259, ExtVal: 240
Train samples: 1473
Test samples: 259
External validation samples: 240
Applying L1 filter on modality position 1
Selected 3636 outlier-stable features
Training 125 epochs with 6 batches
Final output - min: -0.99

# Youden Calibration

The **Youden Index** (J) is used to determine the optimal classification threshold for converting continuous risk scores into binary predictions (responder/non-responder).

**Formula:**
```
J = Sensitivity + Specificity - 1 = TPR - FPR
```

Where:
- **Sensitivity (TPR)** = True Positive Rate (correctly identified positives)
- **Specificity** = True Negative Rate (correctly identified negatives)  
- **FPR** = False Positive Rate = 1 - Specificity

**Interpretation:**
The Youden index finds the point on the ROC curve that maximizes the vertical distance from the diagonal (random classifier line). This balances sensitivity and specificity, identifying the threshold that best discriminates between the two classes.

**Unit Variance Scaling:**
After finding the optimal threshold, all risk scores are divided by their standard deviation to normalize the scale. This ensures:
- Scores are comparable across different models or cross-validation folds
- The distribution has variance = 1
- The threshold is also scaled accordingly

This calibration procedure is applied separately for cross-validation results (on all out-of-fold predictions) and standard train/test splits (using only training data to avoid leakage).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, delong_roc_variance, roc_curve

def calibrate_cv_scores(df_cv_results):
    """
    Calibrate CV scores using Youden index and unit variance scaling.
    
    In cross-validation, each patient appears exactly once as test across all folds.
    Therefore, we calibrate globally on all out-of-fold predictions combined,
    since these already represent the complete dataset tested without data leakage.
    
    This is different from standard split where we must calibrate on train only.
    """
    labels = df_cv_results['label'].values
    scores = df_cv_results['score'].values
    
    # AUC before calibration
    auc_before, auc_cov_before = delong_roc_variance(labels, scores)
    auc_std_before = np.sqrt(auc_cov_before)
    ci_lower_before = auc_before - 1.96 * auc_std_before
    ci_upper_before = auc_before + 1.96 * auc_std_before
    
    # Youden on all data
    fpr, tpr, thresholds = roc_curve(labels, scores)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    
    # Predictions before calibration (using median as arbitrary threshold)
    pred_before = (scores > np.median(scores)).astype(int)
    
    # Unit variance
    score_std = np.std(scores)
    scores_scaled = scores / score_std if score_std > 0 else scores
    threshold_scaled = optimal_threshold / score_std if score_std > 0 else optimal_threshold
    
    df_cv_results['score_calibrated'] = scores_scaled
    df_cv_results['prediction'] = (scores_scaled > threshold_scaled).astype(int)
    df_cv_results['youden_threshold'] = threshold_scaled
    
    # Predictions after calibration (using Youden threshold)
    pred_after = df_cv_results['prediction'].values
    
    # AUC after calibration
    auc_after, auc_cov_after = delong_roc_variance(labels, scores_scaled)
    auc_std_after = np.sqrt(auc_cov_after)
    ci_lower_after = auc_after - 1.96 * auc_std_after
    ci_upper_after = auc_after + 1.96 * auc_std_after
    
    # Classification metrics before
    acc_before = accuracy_score(labels, pred_before)
    sens_before = recall_score(labels, pred_before, zero_division=0)
    tn_before, fp_before, fn_before, tp_before = confusion_matrix(labels, pred_before).ravel()
    spec_before = tn_before / (tn_before + fp_before) if (tn_before + fp_before) > 0 else 0
    f1_before = f1_score(labels, pred_before, zero_division=0)
    
    # Classification metrics after
    acc_after = accuracy_score(labels, pred_after)
    sens_after = recall_score(labels, pred_after, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, pred_after).ravel()
    spec_after = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_after = f1_score(labels, pred_after, zero_division=0)
    
    print(f"CV: Youden threshold={optimal_threshold:.3f}, scaled={threshold_scaled:.3f}, std={score_std:.3f}")
    print(f"AUC before: {auc_before:.3f} (95% CI: {ci_lower_before:.3f}-{ci_upper_before:.3f})")
    print(f"AUC after:  {auc_after:.3f} (95% CI: {ci_lower_after:.3f}-{ci_upper_after:.3f})")
    print(f"\nClassification metrics (before vs after):")
    print(f"Accuracy:    {acc_before:.3f} -> {acc_after:.3f}")
    print(f"Sensitivity: {sens_before:.3f} -> {sens_after:.3f}")
    print(f"Specificity: {spec_before:.3f} -> {spec_after:.3f}")
    print(f"F1-score:    {f1_before:.3f} -> {f1_after:.3f}\n")
    
    return df_cv_results, {'auc': auc_after, 'ci_lower': ci_lower_after, 'ci_upper': ci_upper_after, 'threshold': threshold_scaled}


def calibrate_split_scores(df_split_results):
    """
    Calibrate train/test/ext_val split.
    Calculate Youden/std on TRAIN only, apply to test/ext_val.
    """
    # Train data only
    train_mask = df_split_results['set'] == 'train'
    train_labels = df_split_results.loc[train_mask, 'label'].values
    train_scores = df_split_results.loc[train_mask, 'score'].values
    
    # Youden on train
    fpr, tpr, thresholds = roc_curve(train_labels, train_scores)
    youden_index = tpr - fpr
    optimal_idx = np.argmax(youden_index)
    optimal_threshold = thresholds[optimal_idx]
    
    # Unit variance on train
    score_std = np.std(train_scores)
    threshold_scaled = optimal_threshold / score_std if score_std > 0 else optimal_threshold
    
    # Apply to ALL sets
    df_split_results['score_calibrated'] = df_split_results['score'] / score_std if score_std > 0 else df_split_results['score']
    df_split_results['prediction'] = (df_split_results['score_calibrated'] > threshold_scaled).astype(int)
    df_split_results['youden_threshold'] = threshold_scaled
    
    print(f"Split: Youden threshold={optimal_threshold:.3f}, scaled={threshold_scaled:.3f}, std={score_std:.3f}\n")
    
    # DeLong AUC for each set (before and after)
    metrics = {}
    for set_name in ['test', 'ext_val']:
        set_mask = df_split_results['set'] == set_name
        if set_mask.sum() > 0:
            set_labels = df_split_results.loc[set_mask, 'label'].values
            set_scores = df_split_results.loc[set_mask, 'score'].values
            set_scores_calibrated = df_split_results.loc[set_mask, 'score_calibrated'].values
            set_predictions = df_split_results.loc[set_mask, 'prediction'].values
            
            # Before
            auc_before, auc_cov_before = delong_roc_variance(set_labels, set_scores)
            auc_std_before = np.sqrt(auc_cov_before)
            ci_lower_before = auc_before - 1.96 * auc_std_before
            ci_upper_before = auc_before + 1.96 * auc_std_before
            
            pred_before = (set_scores > np.median(set_scores)).astype(int)
            
            # After
            auc_after, auc_cov_after = delong_roc_variance(set_labels, set_scores_calibrated)
            auc_std_after = np.sqrt(auc_cov_after)
            ci_lower_after = auc_after - 1.96 * auc_std_after
            ci_upper_after = auc_after + 1.96 * auc_std_after
            
            # Classification metrics before
            acc_before = accuracy_score(set_labels, pred_before)
            sens_before = recall_score(set_labels, pred_before, zero_division=0)
            tn_before, fp_before, fn_before, tp_before = confusion_matrix(set_labels, pred_before).ravel()
            spec_before = tn_before / (tn_before + fp_before) if (tn_before + fp_before) > 0 else 0
            f1_before = f1_score(set_labels, pred_before, zero_division=0)
            
            # Classification metrics after
            acc_after = accuracy_score(set_labels, set_predictions)
            sens_after = recall_score(set_labels, set_predictions, zero_division=0)
            tn, fp, fn, tp = confusion_matrix(set_labels, set_predictions).ravel()
            spec_after = tn / (tn + fp) if (tn + fp) > 0 else 0
            f1_after = f1_score(set_labels, set_predictions, zero_division=0)
            
            metrics[set_name] = {
                'auc_before': auc_before, 'ci_lower_before': ci_lower_before, 'ci_upper_before': ci_upper_before,
                'auc_after': auc_after, 'ci_lower_after': ci_lower_after, 'ci_upper_after': ci_upper_after
            }
            
            print(f"{set_name.upper()}:")
            print(f"AUC before: {auc_before:.3f} (95% CI: {ci_lower_before:.3f}-{ci_upper_before:.3f})")
            print(f"AUC after:  {auc_after:.3f} (95% CI: {ci_lower_after:.3f}-{ci_upper_after:.3f})")
            print(f"Classification metrics (before vs after):")
            print(f"Accuracy:    {acc_before:.3f} -> {acc_after:.3f}")
            print(f"Sensitivity: {sens_before:.3f} -> {sens_after:.3f}")
            print(f"Specificity: {spec_before:.3f} -> {spec_after:.3f}")
            print(f"F1-score:    {f1_before:.3f} -> {f1_after:.3f}\n")
    
    return df_split_results, metrics

In [50]:
# Per risultati CV (LOCO-fold)
df_cv = pd.read_csv('results/cv_rwd_cohort2_CBR.csv', index_col=0)
df_cv_calibrated, cv_metrics = calibrate_cv_scores(df_cv)
df_cv_calibrated.to_csv('results/cv_rwd_cohort2_CBR_calibrated.csv')

# Per risultati standard split (train/test/ext_val)
df_split = pd.read_csv('results/standard_rwd_cohort2_CBR.csv', index_col=0)
df_split_calibrated, split_metrics = calibrate_split_scores(df_split)
df_split_calibrated.to_csv('results/standard_rwd_cohort2_CBR_calibrated.csv')

CV: Youden threshold=0.306, scaled=0.304, std=1.005
AUC before: 0.695 (95% CI: 0.663-0.728)
AUC after:  0.695 (95% CI: 0.663-0.728)

Classification metrics (before vs after):
Accuracy:    0.643 -> 0.643
Sensitivity: 0.622 -> 0.584
Specificity: 0.674 -> 0.727
F1-score:    0.672 -> 0.658

Split: Youden threshold=-0.410, scaled=-0.410, std=1.000

TEST:
AUC before: 0.662 (95% CI: 0.579-0.745)
AUC after:  0.662 (95% CI: 0.579-0.745)
Classification metrics (before vs after):
Accuracy:    0.617 -> 0.678
Sensitivity: 0.590 -> 0.915
Specificity: 0.667 -> 0.238
F1-score:    0.667 -> 0.787

EXT_VAL:
AUC before: 0.722 (95% CI: 0.651-0.794)
AUC after:  0.722 (95% CI: 0.651-0.794)
Classification metrics (before vs after):
Accuracy:    0.662 -> 0.624
Sensitivity: 0.633 -> 1.000
Specificity: 0.707 -> 0.037
F1-score:    0.695 -> 0.764



# Scores + images  

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from lung_helpers import delong_roc_variance
import itertools

def calculate_metrics(labels, scores):
    """Calculate AUC, F1, Sensitivity, Specificity"""
    # AUC with DeLong
    auc, auc_cov = delong_roc_variance(labels, scores)
    auc_std = np.sqrt(auc_cov)
    ci_lower = auc - 1.96 * auc_std
    ci_upper = auc + 1.96 * auc_std
    
    # Binary predictions using median threshold (will be replaced by Youden later)
    predictions = (scores > np.median(scores)).astype(int)
    
    # Classification metrics
    f1 = f1_score(labels, predictions, zero_division=0)
    sensitivity = recall_score(labels, predictions, zero_division=0)
    
    tn, fp, fn, tp = confusion_matrix(labels, predictions).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'auc': auc,
        'auc_ci_lower': ci_lower,
        'auc_ci_upper': ci_upper,
        'f1': f1,
        'sensitivity': sensitivity,
        'specificity': specificity
    }

def process_all_results():
    """
    Process all result files and create summary table
    """
    results_dir = Path('results')
    results_dir.mkdir(exist_ok=True)
    
    # Define all combinations
    training_types = ['cv', 'standard']
    modalities = ['rwd', 'rwd_dp', 'rwd_pyrad', 'rwd_radfm', 
                  'rwd_pyrad_dp', 'rwd_radfm_dp', 
                  'rwd_radfm_dp_genomics', 'rwd_pyrad_dp_genomics']
    cohorts = ['cohort23', 'cohort2']
    outcomes = ['OS_6', 'OS_24', 'DCR', 'ORR', 'CBR']

    # if you want to calculate for calibrated scores, just add '_{calibrated}' at the end of filepath variable, right before before .csv
    
    all_results = []
    
    for training, modality, cohort, outcome in itertools.product(
        training_types, modalities, cohorts, outcomes
    ):
        filepath = results_dir / f'{training}_{modality}_{cohort}_{outcome}.csv'
        
        if not filepath.exists():
            continue
            
        print(f"Processing: {filepath.name}")
        df = pd.read_csv(filepath, index_col=0)
        
        if training == 'cv':
            # CV: single set of results
            labels = df['label'].values
            scores = df['score'].values
            
            metrics = calculate_metrics(labels, scores)
            
            all_results.append({
                'training': training,
                'modality': modality,
                'cohort': cohort,
                'outcome': outcome,
                'set': 'all',
                **metrics
            })
            
        else:  # standard
            # Standard: train/test/ext_val sets
            for set_name in ['train', 'test', 'ext_val']:
                set_mask = df['set'] == set_name
                if set_mask.sum() == 0:
                    continue
                    
                labels = df.loc[set_mask, 'label'].values
                scores = df.loc[set_mask, 'score'].values
                
                metrics = calculate_metrics(labels, scores)
                
                all_results.append({
                    'training': training,
                    'modality': modality,
                    'cohort': cohort,
                    'outcome': outcome,
                    'set': set_name,
                    **metrics
                })
    
    # Create summary dataframe
    summary_df = pd.DataFrame(all_results)
    
    # Save
    output_path = results_dir / 'summary_all_results.csv'
    summary_df.to_csv(output_path, index=False)
    print(f"\nSummary saved to: {output_path}")
    
    return summary_df

# Usage
summary = process_all_results()

Processing: cv_rwd_cohort23_OS_6.csv
Processing: cv_rwd_cohort23_OS_24.csv
Processing: cv_rwd_cohort23_DCR.csv
Processing: cv_rwd_cohort23_ORR.csv
Processing: cv_rwd_cohort23_CBR.csv
Processing: cv_rwd_cohort2_OS_6.csv
Processing: cv_rwd_cohort2_OS_24.csv
Processing: cv_rwd_cohort2_DCR.csv
Processing: cv_rwd_cohort2_ORR.csv
Processing: cv_rwd_cohort2_CBR.csv
Processing: cv_rwd_dp_cohort23_OS_6.csv
Processing: cv_rwd_dp_cohort23_OS_24.csv
Processing: cv_rwd_dp_cohort23_DCR.csv
Processing: cv_rwd_dp_cohort23_ORR.csv
Processing: cv_rwd_dp_cohort23_CBR.csv
Processing: cv_rwd_dp_cohort2_OS_6.csv
Processing: cv_rwd_dp_cohort2_OS_24.csv
Processing: cv_rwd_dp_cohort2_DCR.csv
Processing: cv_rwd_dp_cohort2_ORR.csv
Processing: cv_rwd_dp_cohort2_CBR.csv
Processing: cv_rwd_pyrad_cohort23_OS_6.csv
Processing: cv_rwd_pyrad_cohort23_OS_24.csv
Processing: cv_rwd_pyrad_cohort23_DCR.csv
Processing: cv_rwd_pyrad_cohort23_ORR.csv
Processing: cv_rwd_pyrad_cohort23_CBR.csv
Processing: cv_rwd_pyrad_cohort2_OS

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def plot_vanguri_results(summary_csv='results/summary_all_results.csv', output_dir='results/plots'):
    """
    Generate line plots from Vanguri results summary
    
    Args:
        summary_csv: Path to summary CSV file
        output_dir: Directory to save plots
    """
    # Create output directory
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    
    # Load summary
    df = pd.read_csv(summary_csv)
    
    # Define modality display names and order
    modality_order = [
        'rwd',
        'rwd_dp', 
        'rwd_pyrad',
        'rwd_radfm',
        'rwd_pyrad_dp',
        'rwd_radfm_dp',
        'rwd_pyrad_dp_genomics',
        'rwd_radfm_dp_genomics'
    ]
    
    modality_labels = {
        'rwd': 'RWD',
        'rwd_dp': 'RWD\nDP',
        'rwd_pyrad': 'RWD\nPYRAD',
        'rwd_radfm': 'RWD\nFM RAD',
        'rwd_pyrad_dp': 'RWD\nDP\nPYRAD',
        'rwd_radfm_dp': 'RWD\nDP\nFM RAD',
        'rwd_pyrad_dp_genomics': 'RWD\nDP\nPYRAD\nGenomics',
        'rwd_radfm_dp_genomics': 'RWD\nDP\nFM RAD\nGenomics'
    }
    
    # Get unique combinations
    cohorts = df['cohort'].unique()
    outcomes = df['outcome'].unique()
    training_types = df['training'].unique()
    
    for cohort in cohorts:
        for outcome in outcomes:
            for training in training_types:
                # Filter data based on training type
                if training == 'cv':
                    subset = df[
                        (df['cohort'] == cohort) &
                        (df['outcome'] == outcome) &
                        (df['training'] == training) &
                        (df['set'] == 'all')
                    ]
                    
                    if len(subset) == 0:
                        continue
                    
                    _create_plot(subset, modality_order, modality_labels, 
                               cohort, outcome, training, 'all', output_dir)
                    
                else:  # standard - plot for each set
                    for set_type in ['test', 'ext_val']:
                        subset = df[
                            (df['cohort'] == cohort) &
                            (df['outcome'] == outcome) &
                            (df['training'] == training) &
                            (df['set'] == set_type)
                        ]
                        
                        if len(subset) == 0:
                            continue
                        
                        _create_plot(subset, modality_order, modality_labels, 
                                   cohort, outcome, training, set_type, output_dir)
    
    print(f"\n✅ All plots saved to: {output_dir}")


def _create_plot(subset, modality_order, modality_labels, cohort, outcome, training, set_name, output_dir):
    """Helper function to create individual plot"""
    
    # Filter and order modalities
    plot_data = []
    for mod in modality_order:
        mod_data = subset[subset['modality'] == mod]
        if len(mod_data) > 0:
            plot_data.append({
                'modality': mod,
                'label': modality_labels[mod],
                'auc': mod_data['auc'].iloc[0],
                'ci_lower': mod_data['auc_ci_lower'].iloc[0],
                'ci_upper': mod_data['auc_ci_upper'].iloc[0]
            })
    
    if len(plot_data) == 0:
        return
    
    # Extract values
    labels = [d['label'] for d in plot_data]
    aucs = np.array([d['auc'] for d in plot_data])
    ci_lowers = np.array([d['ci_lower'] for d in plot_data])
    ci_uppers = np.array([d['ci_upper'] for d in plot_data])
    
    # Create plot
    plt.figure(figsize=(12, 6))
    
    # Title
    if training == 'cv':
        plot_title = f'CV AUC - {cohort} - {outcome}'
        metric_label = 'CV AUC'
    else:
        plot_title = f'{set_name.upper()} AUC - {cohort} - {outcome}'
        metric_label = f'{set_name.upper()} AUC'
    
    # Plot line with confidence intervals
    plt.plot(labels, aucs, marker='o', linestyle='-', color='#1a80bb', label=metric_label, linewidth=2)
    
    if len(labels) == 1:
        plt.errorbar(labels, aucs, yerr=[aucs - ci_lowers, ci_uppers - aucs], 
                    fmt='o', color='#1a80bb', capsize=5)
    else:
        plt.fill_between(range(len(labels)), ci_lowers, ci_uppers, 
                        color='#8cc5e3', alpha=0.3, label='95% CI')
    
    # Add value labels
    for i, (val, lower, upper) in enumerate(zip(aucs, ci_lowers, ci_uppers)):
        ci_range = val - lower
        plt.text(i, 0.02, f"{val:.3f}\n±{ci_range:.3f}", fontsize=9, ha='center', color='#1a80bb')
    
    plt.title(plot_title, pad=20, fontsize=14)
    plt.ylabel('AUC', fontsize=12)
    plt.ylim(0, 1)
    plt.xticks(range(len(labels)), labels, fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=10)
    
    # Save plot
    filename = f'auc_{training}_{set_name}_{cohort}_{outcome}.png'
    plt.savefig(output_dir / filename, dpi=600, bbox_inches='tight')
    print(f"Saved: {filename}")
    
    # Save data as CSV
    plot_df = pd.DataFrame({
        'modality': [d['modality'] for d in plot_data],
        'auc': aucs,
        'ci_lower': ci_lowers,
        'ci_upper': ci_uppers
    })
    csv_filename = f'auc_{training}_{set_name}_{cohort}_{outcome}.csv'
    plot_df.to_csv(output_dir / csv_filename, index=False)
    
    plt.close()


# Usage
plot_vanguri_results()

findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic f

findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic f

Saved: auc_cv_all_cohort23_OS_6.png


findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic f

Saved: auc_standard_test_cohort23_OS_6.png


...

Saved: auc_cv_all_cohort2_CBR.png


findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic f

Saved: auc_standard_test_cohort2_CBR.png


findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: helvetica
findfont: Generic f

Saved: auc_standard_ext_val_cohort2_CBR.png

✅ All plots saved to: results/plots
